# 🚚 Transportation Optimization Masterclass
This notebook represents the ultimate, mathematically rigorous solver for the Transportation Problem.

### 🌟 Features:
1. **End-to-End Solver**: Automatically handles VAM (initial BFS) $\rightarrow$ MODI (Optimization).
2. **Native Maximization**: Flawlessly solves profit maximization problems (`mode='max'`).
3. **Prohibited Routes**: Safely routes around blocked cells (Input `'X'`, `'M'`, or `None`).
4. **Step-by-Step Visualization**: Set `verbose=True` to watch the exact closed loops, $u_i/v_j$ values, and tie-breakers in real-time.

---

### 📜 Master Tie-Breaking Rules Implemented
This code strictly adheres to the mathematical tie-breakers required for robust OR solutions:

**VAM Initialization:**
1. **Penalty Tie:** Compares tied penalties and selects the one with the *lower minimum cost*.
2. **Penalty & Min Cost Tie:** Selects the row/column that allows the *largest possible allocation* ($\min(\text{supply}, \text{demand})$).
3. **Simultaneous Exhaustion:** If a row and column hit $0$ demand simultaneously, the algorithm elegantly preserves the spanning tree ($m+n-1$) by keeping one active with an $\epsilon$ zero-demand.

**MODI Optimization:**
1. **Multiple Negative $\Delta$ Tie:** If multiple empty cells have the exact same negative penalty, the code dynamically simulates the loops for all tied cells and selects the one offering the **largest $\theta$** (maximum immediate cost reduction).
2. **$\Delta = 0$ Check:** Correctly identifies Alternative Optimal Solutions without triggering infinite loops.
3. **Leaving Cell Tie (Degeneracy):** If multiple subtract cells in a loop tie for $\theta$, only **one** is removed from the basis. The others remain safely in the basic cells list as $0.0$ allocations ($\epsilon$), mathematically preventing matrix collapse.


In [ ]:
import numpy as np
import pandas as pd

class TransportationSolver:
    def __init__(self, costs, supply, demand, mode='min'):
        self.mode = mode.lower()
        self.supply = np.array(supply, dtype=float)
        self.demand = np.array(demand, dtype=float)
        
        # 1. Parse Costs and Prohibited Routes (M / X)
        parsed = []
        for row in costs:
            r = []
            for val in row:
                if str(val).upper() in ['X', 'M', 'NONE', 'INF']: r.append(np.inf)
                else: r.append(float(val))
            parsed.append(r)
        self.costs = np.array(parsed)
        
        self.num_rows, self.num_cols = len(self.supply), len(self.demand)
        
        # 2. Balance the Problem (Dummy Cost/Profit = 0)
        self._balance_problem()
        
        # Save original parsed costs for final calculation
        self.original_costs = self.costs.copy()
        
        # 3. Handle Maximization (Profit -> Cost conversion)
        if self.mode == 'max':
            valid_costs = self.costs[self.costs != np.inf]
            max_c = np.max(valid_costs) if len(valid_costs) > 0 else 0
            self.costs = max_c - self.costs
            self.costs[self.costs == -np.inf] = np.inf # Restore prohibited routes
            
        self.tol = 1e-5

    def _balance_problem(self):
        total_sup, total_dem = np.sum(self.supply), np.sum(self.demand)
        if total_sup > total_dem:
            self.demand = np.append(self.demand, total_sup - total_dem)
            self.costs = np.column_stack((self.costs, np.zeros(self.num_rows)))
            self.num_cols += 1
        elif total_dem > total_sup:
            self.supply = np.append(self.supply, total_dem - total_sup)
            self.costs = np.vstack((self.costs, np.zeros(self.num_cols)))
            self.num_rows += 1

    def solve(self, verbose=False):
        if verbose: print("--- STARTING VAM ---")
        self._run_vam(verbose)
        if verbose: print("\n--- STARTING MODI ---")
        return self._run_modi(verbose)

    def _run_vam(self, verbose):
        sup_c, dem_c = self.supply.copy(), self.demand.copy()
        self.allocation = np.zeros((self.num_rows, self.num_cols))
        self.basic_cells = []
        active_rows, active_cols = list(range(self.num_rows)), list(range(self.num_cols))

        def get_metrics(is_row, idx, active_other):
            c = self.costs[idx, active_other] if is_row else self.costs[active_other, idx]
            sorted_idx = np.argsort(c)
            min_cost = c[sorted_idx[0]]
            pen = c[sorted_idx[1]] - min_cost if len(c) > 1 else min_cost
            
            # Tie breaker: max allocation among tied min costs
            min_others = [active_other[i] for i in sorted_idx if c[i] == min_cost]
            best_alloc, best_other = -1, min_others[0]
            for other in min_others:
                alloc = min(sup_c[idx], dem_c[other]) if is_row else min(sup_c[other], dem_c[idx])
                if alloc > best_alloc:
                    best_alloc = alloc
                    best_other = other
                    
            # Tuple priority: 1. Max Penalty, 2. Min Cost (represented as -min_cost), 3. Max Allocation
            return (pen, -min_cost, best_alloc, idx, best_other)

        while active_rows and active_cols:
            row_metrics = [get_metrics(True, r, active_cols) for r in active_rows]
            col_metrics = [get_metrics(False, c, active_rows) for c in active_cols]
            
            best_row, best_col = max(row_metrics), max(col_metrics)
            
            if best_row >= best_col:
                r_idx, c_idx = best_row[3], best_row[4]
            else:
                c_idx, r_idx = best_col[3], best_col[4]
                
            qty = min(sup_c[r_idx], dem_c[c_idx])
            self.allocation[r_idx, c_idx] = qty
            self.basic_cells.append((r_idx, c_idx))
            sup_c[r_idx] -= qty
            dem_c[c_idx] -= qty
            
            # VAM Exhaustion Tie-Breaker (Maintain Spanning Tree)
            if sup_c[r_idx] == 0 and dem_c[c_idx] == 0:
                if len(active_rows) > 1: active_rows.remove(r_idx)
                elif len(active_cols) > 1: active_cols.remove(c_idx)
                else:
                    active_rows.remove(r_idx)
                    active_cols.remove(c_idx)
            elif sup_c[r_idx] == 0: active_rows.remove(r_idx)
            else: active_cols.remove(c_idx)
                
        # Fill remaining required basics for extreme degeneracy
        req_basic = self.num_rows + self.num_cols - 1
        for r in range(self.num_rows):
            for c in range(self.num_cols):
                if len(self.basic_cells) >= req_basic: break
                if (r, c) not in self.basic_cells: self.basic_cells.append((r, c))

    def _get_loop(self, start_r, start_c):
        cells = self.basic_cells + [(start_r, start_c)]
        while True:
            r_counts = {r: sum(1 for x, y in cells if x == r) for r in range(self.num_rows)}
            c_counts = {c: sum(1 for x, y in cells if y == c) for c in range(self.num_cols)}
            to_remove = [(r, c) for r, c in cells if r_counts[r] == 1 or c_counts[c] == 1]
            if not to_remove: break
            for cell in to_remove: cells.remove(cell)
                
        if not cells: return []
        loop = [(start_r, start_c)]
        cells.remove((start_r, start_c))
        is_row_move = True
        
        while cells:
            curr_r, curr_c = loop[-1]
            next_cell = next(((r, c) for r, c in cells if (is_row_move and r == curr_r) or (not is_row_move and c == curr_c)), None)
            if next_cell:
                loop.append(next_cell)
                cells.remove(next_cell)
                is_row_move = not is_row_move
            else: break
        return loop

    def _run_modi(self, verbose):
        iteration = 0
        while True:
            iteration += 1
            if iteration > 100: raise Exception("Infinite Loop Detected")
            
            u, v = {r: None for r in range(self.num_rows)}, {c: None for c in range(self.num_cols)}
            u[0] = 0 
            
            changed = True
            while changed:
                changed = False
                for r, c in self.basic_cells:
                    if u[r] is not None and v[c] is None:
                        v[c] = self.costs[r, c] - u[r]; changed = True
                    elif v[c] is not None and u[r] is None:
                        u[r] = self.costs[r, c] - v[c]; changed = True
            
            for i in range(self.num_rows):
                if u[i] is None: u[i] = 0
            for j in range(self.num_cols):
                if v[j] is None: v[j] = 0
                
            min_pen = -self.tol
            candidates = []
            for r in range(self.num_rows):
                for c in range(self.num_cols):
                    if (r, c) not in self.basic_cells:
                        pen = self.costs[r, c] - (u[r] + v[c])
                        if pen < min_pen:
                            min_pen = pen
                            candidates = [(r, c)]
                        elif abs(pen - min_pen) <= self.tol:
                            candidates.append((r, c))
                            
            if not candidates:
                status = "Optimal"
                for r in range(self.num_rows):
                    for c in range(self.num_cols):
                        if (r, c) not in self.basic_cells and abs(self.costs[r, c] - (u[r] + v[c])) <= self.tol:
                            status = "Multiple Optimal Solutions"
                            
                total_cost = sum(self.allocation[r, c] * self.original_costs[r, c] for r, c in self.basic_cells if self.allocation[r, c] > 0)
                if verbose:
                    print(f"Final Status: {status}")
                    print(f"Final Total Cost/Profit: {total_cost}")
                return {"status": status, "allocation": self.allocation, "cost": total_cost, "iterations": iteration}
                
            # Tie breaker: Larger Theta
            if len(candidates) > 1:
                best_theta, best_cell = -1, candidates[0]
                for r, c in candidates:
                    loop = self._get_loop(r, c)
                    if loop:
                        sub_cells = [loop[i] for i in range(1, len(loop), 2)]
                        theta = min([self.allocation[x[0], x[1]] for x in sub_cells], default=0)
                        if theta > best_theta:
                            best_theta, best_cell = theta, (r, c)
                enter_r, enter_c = best_cell
                if verbose: print(f"Iter {iteration}: Tie in negative delta! Selected {best_cell} for larger theta.")
            else:
                enter_r, enter_c = candidates[0]
                
            loop = self._get_loop(enter_r, enter_c)
            sub_cells = [loop[i] for i in range(1, len(loop), 2)]
            add_cells = [loop[i] for i in range(0, len(loop), 2)]
            
            min_theta = min(self.allocation[r, c] for r, c in sub_cells)
            leaving_candidates = [(r, c) for r, c in sub_cells if self.allocation[r, c] == min_theta]
            
            theta_cell = leaving_candidates[0]
            if verbose:
                print(f"Iter {iteration}: Enter ({enter_r}, {enter_c}), Theta = {min_theta}")
                if len(leaving_candidates) > 1:
                    print(f"    [TIE] Multiple cells drop to 0! Removing {theta_cell}, keeping others as epsilon.")
            
            for r, c in sub_cells: self.allocation[r, c] -= min_theta
            for r, c in add_cells: self.allocation[r, c] += min_theta
                
            self.basic_cells.remove(theta_cell)
            self.basic_cells.append((enter_r, enter_c))


### 🧪 The Ultimate 8 Edge-Case Test Suite
Run this cell to test the solver against every possible tie-breaker and edge case.


In [ ]:
problems = {
    "1. Standard Balanced (VAM Tie)": {
        "c": [[2, 3, 4], [5, 2, 1], [3, 6, 2]],
        "s": [10, 15, 20], "d": [15, 15, 15], "m": "min"
    },
    "2. Initial Degeneracy": {
        "c": [[4, 8, 8], [6, 4, 5], [8, 7, 6]],
        "s": [20, 30, 25], "d": [25, 30, 20], "m": "min"
    },
    "3. Alternative Optima": {
        "c": [[3, 1, 7, 4], [2, 6, 5, 9], [8, 3, 3, 2]],
        "s": [30, 40, 20], "d": [20, 30, 15, 25], "m": "min"
    },
    "4. Unbalanced (Dummy req.)": {
        "c": [[4, 6, 8, 7], [5, 3, 7, 6], [6, 4, 5, 8]],
        "s": [25, 35, 30], "d": [20, 25, 30, 25], "m": "min"
    },
    "5. Deep MODI Loop": {
        "c": [[6, 20, 8], [10, 7, 20], [5, 5, 4]],
        "s": [20, 30, 10], "d": [10, 25, 25], "m": "min"
    },
    "6. Maximization (Profit)": {
        "c": [[12, 10, 15, 8], [9, 14, 11, 13], [10, 8, 16, 12]],
        "s": [30, 40, 20], "d": [20, 25, 20, 25], "m": "max"
    },
    "7. Prohibited Route (M)": {
        "c": [[5, 8, 'M'], [7, 2, 4]],
        "s": [20, 30], "d": [15, 20, 15], "m": "min"
    },
    "8. Mid-MODI Degeneracy": {
        "c": [[2, 4, 3], [3, 2, 5], [5, 3, 2]],
        "s": [10, 20, 20], "d": [15, 15, 20], "m": "min"
    }
}

for name, data in problems.items():
    print(f"\n{'='*40}\nRunning: {name}")
    solver = TransportationSolver(data["c"], data["s"], data["d"], mode=data["m"])
    # Turn verbose=True on Problem 5 or 8 to see the inner workings!
    is_verbose = (name == "8. Mid-MODI Degeneracy") 
    res = solver.solve(verbose=is_verbose)
    if not is_verbose:
        print(f"Status: {res['status']}")
        print(f"Final Value: {res['cost']}")
        print(f"Iterations: {res['iterations']}")

